In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#the class for each payment
class Installment :
    def __init__ (self,period,pmt,intrest,principal,previous_balance,new_balance):
        self.period=period
        self.pmt=pmt
        self.total_payment=pmt+85000 #additional cost for each payment
        self.intrest=intrest
        self.principal=principal
        self.previous_balance=previous_balance
        self.new_balance=new_balance
        self.cumulative_intrest=0 #بهره تجمعی
        self.cumulative_principal=0 # اصل تجمعی
        
    def __str__ (self):
        return f'in period:{self.period},pmt={self.pmt},intrest={self.intrest},principal={self.principal},new balance={self.new_balance}\n'
    
    def as_dict(self): 
        return {'period':self.period,
                'previous balance':self.previous_balance,
                'Intrest':self.intrest,
                'Principal':self.principal,
                'new balance':self.new_balance,
                'Cumulative interest':self.cumulative_intrest,
                'Cumulative principal':self.cumulative_principal}



#the class for each loan
class AmortizationSchedule :
    def __init__ (self,p,r,m,n):
        self.p=p-p*0.015      #کسر یک و نیم درصد کارمزد در ابتدای تشکیل پرونده
        self.r=r
        self.m=m #تعداد پرداخت در هر سال, number of payments in a year
        self.n=n #تعداد تمامی پرداخت ها, total number of payments
        self.i=r/m
        self.pmt=p*((self.i*(1+self.i)**n)/((1+self.i)**n-1)) # فرمول محاسبه قسط سالیانه بر حسب مقدار اولیه
        self.schedule=[]

    def create_payment (self,t,balance):
        intrest_t=balance*self.i
        principal_t=self.pmt-intrest_t
        balence_t=balance-principal_t
        return Installment(period=t,pmt=self.pmt,intrest=intrest_t,principal=principal_t,previous_balance=balance,new_balance=balence_t)
    
    
    def create_schedule(self,x):  #در حالت نبود نرخ بهره ی شناور x=n
        b=self.p

        for i in range(1,x+1):
            new_payment=self.create_payment(i,b)
            if i==1:
                new_payment.cumulative_intrest+=new_payment.intrest
                new_payment.cumulative_principal+=new_payment.principal
            else:
                new_payment.cumulative_intrest=self.schedule[i-2].cumulative_intrest+new_payment.intrest #محاسبه بهره تجمعی
                new_payment.cumulative_principal=self.schedule[i-2].cumulative_principal+new_payment.principal #محاسبه اصل تجمعی
            self.schedule.append(new_payment)
            b = new_payment.new_balance

    


    def as_dataframe(self):
        return pd.DataFrame([p.as_dict() for p in self.schedule],columns=[
            'period','previous balance','Intrest','Principal','new balance','Cumulative interest','Cumulative principal'])
    



original scenario:

In [ ]:

loan=AmortizationSchedule(780000000,0.21,12,42)
loan.create_schedule(42)
df=loan.as_dataframe()
df["total payment"] = loan.schedule[0].total_payment
df

In [ ]:
plt.plot(df['period'],df['Intrest'],label="Interest",color='orange')
plt.plot(df['period'],df['Principal'],label="Principal",color='green')
plt.show()

Period extention:

In [ ]:
loan=AmortizationSchedule(780000000,0.21,12,60) #cuz payments will be done in 60 months
loan.create_schedule(60)
df=loan.as_dataframe()
df["total payment"] = loan.schedule[0].total_payment
df

In [ ]:
plt.plot(df['period'],df['Intrest'],label="Interest",color='blue')
plt.plot(df['period'],df['Principal'],label="Principal",color='yellow')
plt.show()

Floating Rate :

In [ ]:
loan=AmortizationSchedule(780000000,0.21,12,42)
loan.create_schedule(18) 
new_p=loan.schedule[17].new_balance 
new_loan=AmortizationSchedule(new_p,0.28,12,24)
new_loan.create_schedule(24)
loan.schedule.extend(new_loan.schedule)
df=loan.as_dataframe()
df["total payment"] = loan.schedule[0].total_payment
df

In [ ]:
plt.plot(df.index.tolist(),df['Intrest'],label="Interest",color='blue')  
plt.plot(df.index.tolist(),df['Principal'],label="Principal",color='red')
# using index instead of periods to have a str8 graph(✿◡‿◡)
plt.show()

Prepayment :

In [ ]:
loan=AmortizationSchedule(780000000,0.21,12,42)
loan.create_schedule(6)

n=7  #شمارنده ای که تعریف کردیم
while loan.schedule[-1].new_balance>0:    
    intrest=loan.schedule[-1].previous_balance*loan.i
    principal=loan.schedule[-1].previous_balance - intrest
    principal+=6500000
    new_balance=loan.schedule[-1].previous_balance - principal
    new_payment=Installment(period=n,pmt=loan.schedule[-1].total_payment,intrest=intrest,principal=principal,
                            previous_balance=loan.schedule[-1].new_balance,new_balance=new_balance)
    new_payment.cumulative_intrest=loan.schedule[-1].cumulative_intrest+new_payment.intrest
    new_payment.cumulative_principal=loan.schedule[-1].cumulative_principal+new_payment.principal
    loan.schedule.append(new_payment)
    n+=1
    if loan.schedule[-1].new_balance==0: 
        loan.schedule[-1].new_balance=0

df=loan.as_dataframe()
df["total payment"] = loan.schedule[0].total_payment
df

In [ ]:
plt.plot(df['period'],df['Intrest'],label="Interest",color='red')
plt.plot(df['period'],df['Principal'],label="Principal",color='green')
plt.show()
#(≧﹏ ≦) I tried but...